# Go2 Track 2 Bonus Project: Colab Starter Notebook

Use this notebook to set up the repo, inspect the interfaces, optionally train or reuse a low-level checkpoint, run single-policy track evaluation, and prepare submission metadata.

It starts from a weak baseline. Your leaderboard submission should train a learned high-level planner for the fixed 5D -> [vx, vy, yaw_rate] interface.


## 1. Configure repository URLs

Leave the default repo URL unless you are working from your own fork. If you rerun setup after editing files, keep `RESET_COURSE_REPO = False` so your local changes are not deleted.


In [1]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/vandanacm/Final-Project-Track-2-Project.git"
COURSE_REPO_BRANCH = "main"
TEAM_NAME = "change_me"
RESET_COURSE_REPO = False
BASE_DIR = Path("/content") if Path("/content").exists() else Path("/tmp/go2_track_bonus_colab")
BASE_DIR.mkdir(parents=True, exist_ok=True)
COURSE_REPO_DIR = BASE_DIR / "go2_track_bonus_repo"

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = BASE_DIR / "mujoco_playground"
UNITREE_DIR = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        if RESET_COURSE_REPO:
            print(f"+ remove existing course repo at {target_dir}")
            shutil.rmtree(target_dir)
        else:
            print(f"+ reuse existing course repo at {target_dir}")
            return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


Running inside Colab.


## 2. Install system packages and clone repositories


In [2]:
import shutil
if shutil.which("ffmpeg") is None:
    if Path("/content").exists():
        run(["apt-get", "update", "-qq"])
        run(["apt-get", "install", "-y", "ffmpeg"])
    else:
        print("[local] ffmpeg is not on PATH; imageio-ffmpeg from requirements is enough for local tests.")
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

import sys
if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError(f"Expected mujoco_playground to be imported from {expected_playground}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
+ git clone https://github.com/google-deepmind/mujoco_playground.git /content/mujoco_playground
+ git -C /content/mujoco_playground fetch --all --tags
+ git -C /content/mujoco_playground checkout dd38c285c6d54266287081e516109f0b15985818
+ git clone https://github.com/unitreerobotics/unitree_mujoco.git /content/unitree_mujoco
+ git -C /content/unitree_mujoco fetch --all --tags
+ git -C /content/unitree_mujoco checkout 1a37b051a10be723405b7ed6dc839361af036d88
+ git clone https://github.com/vandanacm/Final-Proje

## 3. Read the assignment requirements

Skim the short specs before editing.


In [3]:
%cd {COURSE_REPO_DIR}
!sed -n '1,180p' docs/assignment_requirements.md
!sed -n '1,140p' docs/controller_interface.md
!sed -n '1,120p' docs/high_level_optimization_guide.md


/content/go2_track_bonus_repo
# Track 2 Requirements

Goal: run Go2 as far as possible around a 200 m oval track in MuJoCo.

## Options

- Proposal-based final project.
- Go2 oval-track leaderboard route.
- Both, for bonus.

## Leaderboard Route

```text
5D track observation -> [vx, vy, yaw_rate] -> Go2 low-level policy
```

Use `docs/controller_interface.md`. The low-level checkpoint should stay
compatible with the HW1 Brax PPO format.
This repo evaluates one submission at a time; ranking compares submitted
outputs.

Leaderboard submissions must train a learned high-level planner for this fixed
interface. The provided starter planner is only a weak baseline for debugging.
Keep the 5D input and 3D output fixed; change the planner internals. The
official scene is fixed: 200 m centerline, 18.25 m turn radius, and 2.0 m half
width. Do not change the track geometry to improve score.

## Allowed

- Reuse a HW1 checkpoint.
- Retrain or modify the low-level Go2 policy.
- Train a learned high-

## 4. Copy Go2 assets


In [4]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


/content/go2_track_bonus_repo
Copied 16 assets into /content/go2_track_bonus_repo/go2_pg_env/xmls/assets


## 5. Inspect the low-level Go2 environment


In [5]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2


/content/go2_track_bonus_repo
{
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_2",
  "backend_impl": "jax",
  "control_dt": 0.02,
  "sim_dt": 0.004,
  "episode_length": 1000,
  "action_size": 12,
  "actor_obs_size": 48,
  "critic_obs_size": 123,
  "observation_layout": {
    "state": [
      [
        "local_linvel",
        3
      ],
      [
        "gyro",
        3
      ],
      [
        "gravity",
        3
      ],
      [
        "joint_pos_error",
        12
      ],
      [
        "joint_vel",
        12
      ],
      [
        "last_action",
        12
      ],
      [
        "command",
        3
      ]
    ],
    "privileged_state_extra": [
      [
        "gyro_clean",
        3
      ],
      [
        "accelerometer",
        3
      ],
      [
        "gravity_clean",
        3
      ],
      [
        "local_linvel_clean",
        3
      ],
      [
        "global_angvel",
        3
      ],
      [
        "joint_pos_error_clean",
       

## 6. Read the important starter files


In [6]:
!sed -n '1,180p' go2_pg_env/joystick.py
!sed -n '1,220p' go2_pg_env/track.py
!sed -n '1,220p' track_bonus/controller_interface.py
!sed -n '1,220p' track_bonus/planner.py
!sed -n '1,220p' run_track_bonus.py


"""Joystick locomotion task for the local Go2 environment.

This task is adapted from MuJoCo Playground's Go1 joystick task. The local
changes are intentionally small so that students can compare the official
baseline against a course-specific Go2 variant.

Observation summary
-------------------
state (actor input):
    [local_linvel(3), gyro(3), gravity(3),
     joint_pos_error(12), joint_vel(12),
     last_action(12), command(3)]  -> 48 dims

privileged_state (critic-only input during training):
    state + extra simulator-only signals -> 123 dims

Action summary
--------------
The policy outputs 12 joint offsets. The final motor target is:
    target_joint_pos = default_pose + action_scale * policy_action
"""

from __future__ import annotations

from typing import Any, Dict, Optional, Union

import jax
import jax.numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco.mjx._src import math
import numpy as np

from mujoco_playground._src import mjx_env



## 7. Define a Colab-friendly low-level training config

This is a normal Colab training starting point, not a quick test. Reduce the step counts for experiments if needed.


In [7]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 18_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config

# Apply the student_stage2_goal ranges so the policy learns vy + yaw tracking.
# Without this the default stage_2 trains forward-only (vy=0, yaw=0) and the
# race planner's turning commands are ignored, resulting in near-zero movement.
goal = base_config["stage_2"]["student_stage2_goal"]
base_config["stage_2"]["command_range"] = goal["command_range"]
base_config["stage_2"]["command_keep_prob"] = goal["command_keep_prob"]

config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)
print("stage_2 command_range:", base_config["stage_2"]["command_range"])
print("stage_2 command_keep_prob:", base_config["stage_2"]["command_keep_prob"])


wrote /content/go2_track_bonus_repo/configs/colab_runtime_config.json
stage_2 command_range: {'min': [-0.6, -0.3, -1.0], 'max': [2.5, 0.3, 1.0]}
stage_2 command_keep_prob: [1.0, 0.4, 0.6]


## 8. Dry-run training config


In [8]:
!python train.py --config configs/colab_runtime_config.json --dry-run


{
  "homework_name": "Final Project Track 2 Bonus: Go2 Low-Level Locomotion Baseline",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "framework": "MuJoCo Playground + Brax PPO + MJX",
  "backend_impl": "jax",
  "actor_obs_key": "state",
  "critic_obs_key": "privileged_state",
  "use_domain_randomization": true,
  "seed": 0,
  "control": {
    "ctrl_dt": 0.02,
    "sim_dt": 0.004,
    "action_scale": 0.5,
    "action_type": "absolute_joint_position_target",
    "torque_mapping": "position_target_through_pd_actuator"
  },
  "course_budget": {
    "baseline_total_env_steps": 15000000,
    "leaderboard_max_env_steps": 30000000,
    "flat_terrain_only": true,
    "require_colab_gpu_runtime": true
  },
  "training_defaults": {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [
      256,
      256,
      128
    ],
    "value_hidden_layer_sizes": [
      256,
      256,
      128
    ]
  },
  "st

## 9. Train a low-level checkpoint from scratch

 The cell below **trains a fresh low-level Go2 policy** and writes `artifacts/low_level_train/best_checkpoint` (the same HW1-compatible Brax PPO actor: `policy_obs_key="state"`, 48-dim obs, 12 actions).

**Sub-90 s plan:** training uses the race ranges in `configs/course_config.json` (stage_2 goal: vx up to 3.0, nonzero vy/yaw, stronger `tracking_ang_vel`), so the policy learns to track the fast turning commands the race planner emits. Budget: 24 M of 30 M steps (a few hours on a Colab T4). For a quick end-to-end smoke run, lower the step counts in Step 7.

The cell only trains if no checkpoint exists yet, so reruns won't restart training.


In [9]:
# Train a fresh low-level Go2 locomotion checkpoint from scratch (no HW1 checkpoint needed).
# This produces the same HW1-compatible Brax PPO actor (policy_obs_key="state", 48-dim
# observation, 12 joint actions) using the race command ranges in course_config.json.
LOW_LEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train"
CHECKPOINT_DIR = LOW_LEVEL_DIR / "best_checkpoint"

if not CHECKPOINT_DIR.exists():
    print("No checkpoint found. Training a fresh low-level policy (this can take a few hours on a Colab GPU)...")
    !cd {COURSE_REPO_DIR} && python train.py \
      --config configs/colab_runtime_config.json \
      --stage both \
      --output-dir {LOW_LEVEL_DIR}
else:
    print("Reusing existing checkpoint at", CHECKPOINT_DIR)

# Race planner: learned feedforward + optional MLP residual (leaderboard-ready).
PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "race_planner.json"
# Debug baseline only:
# PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "starter_planner.json"

print("checkpoint path:", CHECKPOINT_DIR)
print("planner config:", PLANNER_CONFIG)
print("checkpoint exists:", CHECKPOINT_DIR.exists())
print("planner exists:", PLANNER_CONFIG.exists())


No checkpoint found. Training a fresh low-level policy (this can take a few hours on a Colab GPU)...
[run] output_dir=/content/go2_track_bonus_repo/artifacts/low_level_train
[run] stages=['stage_1', 'stage_2']
[stage_1] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=10000000 num_envs=1024 batch_size=256 num_evals=5
[stage_1] steps=0 eval_reward=0.045
[stage_1] steps=3276800 eval_reward=21.693
[stage_1] steps=6553600 eval_reward=41.741
[stage_1] steps=9830400 eval_reward=46.561
[stage_1] steps=13107200 eval_reward=49.028
[stage_1] finished: latest_checkpoint=/content/go2_track_bonus_repo/artifacts/low_level_train/stage_1/checkpoints/000013107200 selected_checkpoint_source=/content/go2_track_bonus_repo/artifacts/low_level_train/stage_1/checkpoints/000013107200
[stage_2] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=18000000 num_envs=1024 batch_size=256 num_evals=5
[stage_2] restoring from checkpoint: /content/go2_track_bonus_repo/artifacts/low_level_t

## 10. Run single-policy track evaluation

**Step 10a** runs a fast no-render eval (writes `results.json` + `leaderboard.csv`).
**Step 10b** optionally renders `race.mp4` (slow — run separately so it can't block `results.json`).


In [10]:
TRACK_EVAL_SMOKE_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval_smoke"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

if CHECKPOINT_DIR.exists():
    # Fast no-render eval -> writes results.json + leaderboard.csv (no video overhead).
    print("Running 90s no-render eval (results.json + leaderboard.csv)...")
    !cd {COURSE_REPO_DIR} && python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_DIR} \
      --entry-name {TEAM_NAME} \
      --duration-seconds 90 \
      --no-render
else:
    print("Skipping track eval: CHECKPOINT_DIR does not exist yet.")
    print("Run Step 9 to train a low-level policy first, then rerun this cell.")


Running 90s no-render eval (results.json + leaderboard.csv)...
{
  "output_dir": "/content/go2_track_bonus_repo/artifacts/track_eval",
  "metrics": {
    "lap_completion": 0.8534492093740905,
    "valid_distance_m": 170.68984187481811,
    "finish_time": null,
    "mean_progress_speed": 1.8965537986090901,
    "alive_time": 90.0,
    "fall": false,
    "fall_step": null,
    "boundary_violation": false,
    "boundary_violation_step": null,
    "rms_lateral_error": 0.2295393849946416,
    "max_lateral_error": 0.3275012969970703,
    "min_boundary_margin_m": 1.6724987030029297,
    "energy_proxy": 17.988101959228516,
    "foot_slip_proxy": 1.983791708946228,
    "total_time": 90.0,
    "num_steps": 4500
  },
  "scores": {
    "completion_score": 0.8534492093740905,
    "speed_score": 1.0,
    "line_keeping_score": 1.0,
    "stability_score": 1.0,
    "efficiency_score": 0.4380307790395376,
    "composite_score": 0.9059536831703177
  }
}


## 10a. (Optional) Render race video

Only run this after Step 10 succeeds. Re-runs the same eval but WITH video rendering.
Skip if you just need `results.json` for submission.

In [11]:
if CHECKPOINT_DIR.exists():
    print("Rendering race video (race.mp4)...")
    !cd {COURSE_REPO_DIR} && MUJOCO_GL=egl python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_DIR} \
      --entry-name {TEAM_NAME} \
      --duration-seconds 90 \
      --render-every 10 \
      --render-fps 5 \
      --render-camera-profile showcase

Rendering race video (race.mp4)...
{
  "output_dir": "/content/go2_track_bonus_repo/artifacts/track_eval",
  "metrics": {
    "lap_completion": 0.8550719008130634,
    "valid_distance_m": 171.01438016261267,
    "finish_time": null,
    "mean_progress_speed": 1.9001597795845853,
    "alive_time": 90.0,
    "fall": false,
    "fall_step": null,
    "boundary_violation": false,
    "boundary_violation_step": null,
    "rms_lateral_error": 0.22028436423931805,
    "max_lateral_error": 0.29370689392089844,
    "min_boundary_margin_m": 1.7062931060791016,
    "energy_proxy": 18.00787925720215,
    "foot_slip_proxy": 1.990670084953308,
    "total_time": 90.0,
    "num_steps": 4500
  },
  "scores": {
    "completion_score": 0.8550719008130634,
    "speed_score": 1.0,
    "line_keeping_score": 1.0,
    "stability_score": 1.0,
    "efficiency_score": 0.4377100660994246,
    "composite_score": 0.9066678586708498
  }
}


## 10b. Show race metrics (and play video if rendered)

Reads `track_eval/results.json` from Step 10. If you also ran Step 10a, the video plays inline.

In [12]:
import json
from base64 import b64encode
from IPython.display import HTML, display

results_path = TRACK_EVAL_DIR / "results.json"
if results_path.exists():
    results = json.loads(results_path.read_text())
    m, s = results["metrics"], results["scores"]
    print("lap_completion      :", m["lap_completion"])
    print("finish_time (s)     :", m["finish_time"])
    print("valid_distance_m    :", m["valid_distance_m"])
    print("mean_progress_speed :", round(float(m["mean_progress_speed"]), 3), "m/s")
    print("fall / boundary     :", m["fall"], "/", m["boundary_violation"])
    print("rms_lateral_error   :", round(float(m["rms_lateral_error"]), 3))
    print("composite_score     :", s["composite_score"])
else:
    print("No results.json yet - run Step 10 first.")

video_path = TRACK_EVAL_DIR / "race.mp4"
if video_path.exists():
    print("\nPlaying", video_path)
    data = b64encode(video_path.read_bytes()).decode()
    display(
        HTML(
            f'<video width=720 controls autoplay loop>'
            f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
        )
    )
else:
    print("No race.mp4 yet - run Step 10a to render it (optional).")

lap_completion      : 0.8550719008130634
finish_time (s)     : None
valid_distance_m    : 171.01438016261267
mean_progress_speed : 1.9 m/s
fall / boundary     : False / False
rms_lateral_error   : 0.22
composite_score     : 0.9066678586708498

Playing /content/go2_track_bonus_repo/artifacts/track_eval/race.mp4


## 11. Train your high-level planner

You are expected to train a learned high-level planner. The command below is only a starter parameter search for debugging the loop; replace the planner internals with your own MLP, RL policy, or other trained policy while keeping the same 5D -> [vx, vy, yaw_rate] interface.

Do not change the track geometry. The evaluator uses the fixed official oval for reset, scoring, and rendering.

A valid learned planner has trained parameters, such as MLP weights. Store those weights in your submission and load them from `StarterTrackPlanner.load(planner_config)`.


In [13]:
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_race"

# 1) Initialize learned weights (feedforward scales; add --include-mlp for race_mlp):
!cd {COURSE_REPO_DIR} && python scripts/init_race_planner_weights.py --include-mlp

# 2) Evolve race_ff (fast, do this first):
!cd {COURSE_REPO_DIR} && python train_highlevel_mlp.py \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --planner-type race_ff \
  --config configs/colab_runtime_config.json \
  --output-dir {HIGHLEVEL_DIR}/race_ff \
  --iterations 12 --population 16 --eval-seconds 90

# 3) Optional: evolve race_mlp residual on top of best race_ff weights:
# !cd {COURSE_REPO_DIR} && python train_highlevel_mlp.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --planner-type race_mlp \
#   --init-weights {HIGHLEVEL_DIR}/race_ff/planner_weights.npz \
#   --config configs/colab_runtime_config.json \
#   --output-dir {HIGHLEVEL_DIR}/race_mlp \
#   --iterations 16 --population 16 --eval-seconds 90

# After training, point PLANNER_CONFIG at the best config and rerun Step 10:
PLANNER_CONFIG = HIGHLEVEL_DIR / "race_ff" / "best_planner_config.json"
print("PLANNER_CONFIG updated to:", PLANNER_CONFIG)


Wrote /content/go2_track_bonus_repo/configs/race_planner_weights.npz
iter=0 cand=0 fitness=3.044 lap=0.8550719008130634 dist=171.01438016261267 best=3.044
iter=0 cand=1 fitness=2.669 lap=0.7482510924211792 dist=149.65021848423584 best=3.044
iter=0 cand=2 fitness=2.919 lap=0.8208347803794516 dist=164.1669560758903 best=3.044
iter=0 cand=3 fitness=2.847 lap=0.7896475923542358 dist=157.92951847084714 best=3.044
iter=0 cand=4 fitness=3.364 lap=0.9642441625349395 dist=192.8488325069879 best=3.364
iter=0 cand=5 fitness=2.920 lap=0.8127016751324075 dist=162.5403350264815 best=3.364
iter=0 cand=6 fitness=3.046 lap=0.8562276858621678 dist=171.24553717243356 best=3.364
iter=0 cand=7 fitness=2.971 lap=0.829989811085739 dist=165.9979622171478 best=3.364
iter=0 cand=8 fitness=3.015 lap=0.8451614631888301 dist=169.03229263776603 best=3.364
iter=0 cand=9 fitness=3.182 lap=0.9019295419288604 dist=180.3859083857721 best=3.364
iter=0 cand=10 fitness=2.999 lap=0.8634678053313501 dist=172.69356106627004 b

## 12. Create submission metadata


In [14]:
import json
submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz, if used",
    "high_level_planner_type": "learned",
    "track_eval": "track_eval/results.json",
    "notes": "Briefly describe your low-level training, learned high-level planner, and failed ideas."
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())


{
  "team_name": "change_me",
  "track2_option": "leaderboard",
  "checkpoint_dir": "best_checkpoint",
  "planner_config": "planner_config.json",
  "planner_code": "track_bonus/planner.py",
  "planner_weights": "planner_weights.npz, if used",
  "high_level_planner_type": "learned",
  "track_eval": "track_eval/results.json",
  "notes": "Briefly describe your low-level training, learned high-level planner, and failed ideas."
}


## 13. Final local checklist

This only checks local paths. If `track_eval/results.json` is missing, run the full evaluation command in Step 10.


In [15]:
from pathlib import Path
expected = {
    "checkpoint": CHECKPOINT_DIR,
    "planner_config": PLANNER_CONFIG,
    "submission_json": COURSE_REPO_DIR / "submission.json",
    "track_eval/results.json": TRACK_EVAL_DIR / "results.json",
    "track_eval/leaderboard.csv": TRACK_EVAL_DIR / "leaderboard.csv",
    "track_eval/race.mp4 (optional)": TRACK_EVAL_DIR / "race.mp4",
}
all_ok = True
for label, path in expected.items():
    status = "OK" if path.exists() else "MISSING"
    if status == "MISSING" and "optional" not in label:
        all_ok = False
    print(f"  {status:7s}  {label}")
if all_ok:
    print("\nAll required files present. Ready to bundle (Step 14).")
else:
    print("\nSome required files are MISSING. Run the corresponding steps first.")


  OK       checkpoint
  OK       planner_config
  OK       submission_json
  OK       track_eval/results.json
  OK       track_eval/leaderboard.csv
  OK       track_eval/race.mp4 (optional)

All required files present. Ready to bundle (Step 14).


In [16]:
import shutil
from pathlib import Path

# Collect submission artifacts into one folder, then zip + download.
SUBMISSION_DIR = COURSE_REPO_DIR / "artifacts" / "submission_bundle"
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

def _copy(src: Path, dst_name: str) -> None:
    src = Path(src)
    if not src.exists():
        print("skip (missing):", src)
        return
    dst = SUBMISSION_DIR / dst_name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print("added:", dst_name)

# Low-level checkpoint (HW1-compatible Brax PPO actor)
_copy(CHECKPOINT_DIR, "best_checkpoint")
# Learned high-level planner config + weights
_copy(PLANNER_CONFIG, "planner_config.json")
_copy(PLANNER_CONFIG.parent / "race_planner_weights.npz", "planner_weights.npz")
# Planner code (in case you modified it)
_copy(COURSE_REPO_DIR / "track_bonus" / "planner.py", "planner.py")
# Submission metadata + evaluation results + video
_copy(COURSE_REPO_DIR / "submission.json", "submission.json")
_copy(TRACK_EVAL_DIR / "results.json", "track_eval/results.json")
_copy(TRACK_EVAL_DIR / "leaderboard.csv", "track_eval/leaderboard.csv")
_copy(TRACK_EVAL_DIR / "race.mp4", "track_eval/race.mp4")

# Zip the bundle
zip_base = COURSE_REPO_DIR / "artifacts" / f"{TEAM_NAME}_track2_submission"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=SUBMISSION_DIR))
print("\nBundle:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")

# Download to your machine (Colab). Falls back to saving in Drive if mounted.
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print("Direct download unavailable:", exc)
    drive_dir = Path("/content/drive/MyDrive")
    if drive_dir.exists():
        drive_dst = drive_dir / zip_path.name
        shutil.copy2(zip_path, drive_dst)
        print("Copied to Drive:", drive_dst)
    else:
        print("Mount Drive (from google.colab import drive; drive.mount('/content/drive')) "
              "or download manually from the Files panel:", zip_path)

added: best_checkpoint
added: planner_config.json
skip (missing): /content/go2_track_bonus_repo/artifacts/highlevel_race/race_ff/race_planner_weights.npz
added: planner.py
added: submission.json
added: track_eval/results.json
added: track_eval/leaderboard.csv
added: track_eval/race.mp4

Bundle: /content/go2_track_bonus_repo/artifacts/change_me_track2_submission.zip (2.6 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>